# Bronze — Schema Evolution (Orders)

1. Simulate new source columns: `order_channel`, `customer_segment`
2. Append evolved sample to `globalmart.bronze.orders` with `mergeSchema`
3. Verify old rows have NULLs for new columns; appended rows are populated
4. Run schema validation and log violations to `globalmart.metadata.schema_violation_log`

In [ ]:
import sys

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.ingestion.schema_evolution import (
    SchemaEvolutionConfig,
    ORDERS_BASE_CONTRACT,
    build_evolved_sample_df,
    ingest_evolved_sample,
    validate_schema,
    log_schema_violations,
    build_evolution_report,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = SchemaEvolutionConfig()
print("Target table:", config.orders_table)

In [ ]:
# Baseline validation on original business columns (expect clean)
from pyspark.sql import functions as F

business_df = spark.table(config.orders_table).select(*ORDERS_BASE_CONTRACT.keys())
baseline_violations = validate_schema(
    business_df, ORDERS_BASE_CONTRACT, table_name=config.orders_table
)
baseline_logged = log_schema_violations(spark, baseline_violations, config.violation_log_table)
print(f"Baseline violations: {len(baseline_violations)} (logged {baseline_logged})")

In [ ]:
# Build evolved sample + write CSV to landing volume
evolved_df = build_evolved_sample_df(spark, config)
display(evolved_df.limit(5))

evolved_df.coalesce(1).write.mode("overwrite").option("header", True).csv(config.evolved_csv_dir)
print("Wrote evolved CSV to:", config.evolved_csv_dir)

In [ ]:
# Append with Delta schema evolution
rows_before = spark.table(config.orders_table).count()
rows_appended = ingest_evolved_sample(spark, evolved_df, config)
rows_after = spark.table(config.orders_table).count()
print(f"Before: {rows_before:,} | Appended: {rows_appended:,} | After: {rows_after:,}")

In [ ]:
# Verify NULLs on old rows vs populated on new rows
ch, seg = config.new_columns
orders = spark.table(config.orders_table)

display(
    orders.groupBy(
        F.col(ch).isNull().alias(f"{ch}_is_null"),
        F.col(seg).isNull().alias(f"{seg}_is_null"),
    ).count()
)

In [ ]:
# Validation demo: strict contract missing new columns → log EXTRA violations
evolved_contract = {
    **ORDERS_BASE_CONTRACT,
    config.new_columns[0]: "string",
    config.new_columns[1]: "string",
}
post_violations = validate_schema(
    business_df, evolved_contract, table_name=config.orders_table
)
post_logged = log_schema_violations(spark, post_violations, config.violation_log_table)
print(f"Post-evolution check on old slice: {len(post_violations)} violations (logged {post_logged})")
display(spark.table(config.violation_log_table).orderBy(F.col("logged_at").desc()))

In [ ]:
import json

report = build_evolution_report(spark, config, rows_appended)
report["violation_log_table"] = config.violation_log_table
print(json.dumps(report, indent=2))

try:
    path = save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/schema_evolution_latest.json",
    )
    print(f"\nSaved to: {path}")
except Exception as exc:
    print(f"\nVolume save skipped: {exc}")

## Reflection

Schema evolution lets pipelines accept new source columns without failing batch loads. Risks: silent type widening, nullable drift, downstream models breaking on unexpected columns, and harder governance without explicit contracts and validation.